In [1]:
import pandas as pd
import numpy as np

### 1. Creating Dates (pd.to_datetime)

In [23]:
# Convert a single string
date = pd.to_datetime('2024-05-01')
date

Timestamp('2024-05-01 00:00:00')

In [25]:
# Handling different formats (Day-Month-Year)
euro_date = pd.to_datetime('31/12/2024', dayfirst=True)
euro_date

Timestamp('2024-12-31 00:00:00')

### 2. Extracting Parts of a Date

In [30]:
df = pd.DataFrame({'raw_date': pd.to_datetime(['2024-01-01', '2024-05-15', '2024-12-31'])})

df['year']  = df['raw_date'].dt.year
df['month'] = df['raw_date'].dt.month
df['day_name'] = df['raw_date'].dt.day_name() # "Monday", "Wednesday", etc.
df['is_weekend'] = df['raw_date'].dt.dayofweek > 4

df

,raw_date,year,month,day_name,is_weekend
0,2024-01-01,2024,1,Monday,False
1,2024-05-15,2024,5,Wednesday,False
2,2024-12-31,2024,12,Tuesday,False


### 3. Date Math with Timedelta

In [31]:
# Add 10 days to every date
df['deadline'] = df['raw_date'] + pd.Timedelta(days=10)

# Calculate the difference between two dates
df['duration'] = df['deadline'] - df['raw_date']

df

,raw_date,year,month,day_name,is_weekend,deadline,duration
0,2024-01-01,2024,1,Monday,False,2024-01-11,10 days
1,2024-05-15,2024,5,Wednesday,False,2024-05-25,10 days
2,2024-12-31,2024,12,Tuesday,False,2025-01-10,10 days


### 4. Generating Date Ranges

In [33]:
# Create a range of 10 days starting today
daily_range = pd.date_range(start='2024-01-01', periods=10, freq='D')
print(daily_range)

DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
               '2024-01-09', '2024-01-10'],
              dtype='datetime64[us]', freq='D')


In [34]:
# Create a range of business days (excludes weekends)
biz_days = pd.date_range(start='2024-01-01', end='2024-01-15', freq='B')
biz_days

DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05', '2024-01-08', '2024-01-09', '2024-01-10',
               '2024-01-11', '2024-01-12', '2024-01-15'],
              dtype='datetime64[us]', freq='B')

### Downsampling (Daily to Monthly)

- h - hourly
- W - Weekly
- QE - Quarter end
- min - Minute
- D - Calender day
- ME - Month end
- YE - Year end

- If datetime column isn't the index:
  
df.resample('D', on='your_date_column').sum()

In [8]:
# Create daily data for 2024
dates = pd.date_range('2024-01-01', periods=100, freq='D')
df = pd.DataFrame({'Sales': np.random.randint(10, 100, 100)}, index=dates)

df.head()

,Sales
2024-01-01,90
2024-01-02,96
2024-01-03,77
2024-01-04,74
2024-01-05,93


In [9]:
# Summarize by month
monthly_total = df.resample('ME').sum() # 'ME' stands for Month End
monthly_total

,Sales
2024-01-31,1894
2024-02-29,1800
2024-03-31,1659
2024-04-30,472


### multiple operations at once

In [11]:
df.resample('ME').agg(['sum', 'mean', 'std'])

Sales                      
             sum       mean        std
2024-01-31  1894  61.096774  24.390374
2024-02-29  1800  62.068966  25.650022
2024-03-31  1659  53.516129  29.567855
2024-04-30   472  52.444444  25.564189

### Comparing Product Performance

In [17]:
# Create data: 3 products over 10 days
dates = pd.date_range('2024-01-01', periods=10, freq='D').tolist() * 3
products = ['Apples'] * 10 + ['Bananas'] * 10 + ['Cherries'] * 10
sales = np.random.randint(5, 50, 30)

df = pd.DataFrame({'Date': dates, 'Product': products, 'Sales': sales})

In [19]:
# Sum sales weekly per product and fill missing weeks with 0
comparison = (df.groupby('Product')
                .resample('W', on='Date')['Sales']
                .sum()
                .unstack(level=0) # Pivot products to columns for easy plotting
                .fillna(0))

comparison

Product,Apples,Bananas,Cherries
Date,,,
2024-01-07,187,136,179
2024-01-14,78,97,83


In [20]:
df.set_index('Date', inplace=True)

# This calculates the mean sales per product, per month
monthly_grouped = df.groupby('Product')['Sales'].resample('ME').mean()
monthly_grouped

Product   Date      
Apples    2024-01-31    26.5
Bananas   2024-01-31    23.3
Cherries  2024-01-31    26.2
Name: Sales, dtype: float64